In [1]:
import numpy as np

In [2]:
k = int((3+1)**2 + 0.5)

In [3]:
k

16

In [10]:
import torch
import math

def spline_basis_fw_cpu(pseudo: torch.Tensor, kernel_size: torch.Tensor, 
                        is_open_spline: torch.Tensor, degree: int) -> tuple:
    """
    Forward spline basis computation on CPU.
    
    Args:
        pseudo: Tensor of shape (E, D) - pseudo coordinates
        kernel_size: Tensor of shape (D,) - kernel size per dimension
        is_open_spline: Tensor of shape (D,) - whether spline is open per dimension
        degree: int - degree of the spline basis
    
    Returns:
        tuple: (basis, weight_index)
            - basis: Tensor of shape (E, S) - basis values
            - weight_index: Tensor of shape (E, S) - weight indices
    """
    # Input validation
    assert pseudo.device.type == 'cpu', "pseudo must be on CPU"
    assert kernel_size.device.type == 'cpu', "kernel_size must be on CPU"
    assert is_open_spline.device.type == 'cpu', "is_open_spline must be on CPU"
    
    assert kernel_size.dim() == 1, "kernel_size must be 1D"
    assert pseudo.size(1) == kernel_size.numel(), "pseudo.size(1) must equal kernel_size.numel()"
    assert is_open_spline.dim() == 1, "is_open_spline must be 1D"
    assert pseudo.size(1) == is_open_spline.numel(), "pseudo.size(1) must equal is_open_spline.numel()"
    
    E = pseudo.size(0)  # Number of edge samples
    D = pseudo.size(1)  # Number of dimensions
    S = int(math.pow(degree + 1, D) + 0.5)  # Number of basis functions
    
    # Initialize output tensors
    basis = torch.empty((E, S), dtype=pseudo.dtype, device=pseudo.device)
    weight_index = torch.empty((E, S), dtype=kernel_size.dtype, device=kernel_size.device)
    
    # Convert to numpy for easier data access (or keep as torch operations)
    kernel_size_data = kernel_size.cpu().numpy()
    is_open_spline_data = is_open_spline.cpu().numpy()
    pseudo_data = pseudo.cpu().numpy()
    
    # Main computation loop
    for e in range(E):
        for s in range(S):
            k = s
            print('k initial = ', k)
            wi = 0
            wi_offset = 1
            b = 1.0
            
            for d in range(D):
                k_mod = k % (degree + 1)
                print("k_mod = ", k_mod)
                k //= (degree + 1)
                print("k = ", k)
                
                # Get pseudo value and scale it
                v = pseudo_data[e, d]
                v *= kernel_size_data[d] - degree * is_open_spline_data[d]
                
                # Compute weight index contribution
                wi += (int(v + k_mod) % kernel_size_data[d]) * wi_offset
                wi_offset *= kernel_size_data[d]
                
                # Compute basis function value
                v -= math.floor(v)
                v = basis_function_forward(v, k_mod, degree)
                b *= v
            
            basis[e, s] = b
            weight_index[e, s] = wi
    
    return basis, weight_index


def basis_function_forward(v: float, k_mod: int, degree: int) -> float:
    """
    Forward pass of B-spline basis function.
    
    Args:
        v: float - parameter value in [0, 1)
        k_mod: int - degree modulo (which piece of the piecewise function)
        degree: int - degree of the spline
    
    Returns:
        float - basis function value
    """
    if degree == 1:
        if k_mod == 0:
            return 1.0 - v
        else:  # k_mod == 1
            return v
    
    elif degree == 2:
        if k_mod == 0:
            return 0.5 * v * v - v + 0.5
        elif k_mod == 1:
            return -v * v + v + 0.5
        else:  # k_mod == 2
            return 0.5 * v * v
    
    elif degree == 3:
        if k_mod == 0:
            return (1.0 - v) ** 3 / 6.0
        elif k_mod == 1:
            return (3.0 * v ** 3 - 6.0 * v ** 2 + 4.0) / 6.0
        elif k_mod == 2:
            return (-3.0 * v ** 3 + 3.0 * v ** 2 + 3.0 * v + 1.0) / 6.0
        else:  # k_mod == 3
            return v ** 3 / 6.0
    
    else:
        raise NotImplementedError(f"Degree {degree} not implemented")


# Vectorized version (more efficient for large batches)
def spline_basis_fw_cpu_vectorized(pseudo: torch.Tensor, kernel_size: torch.Tensor, 
                                   is_open_spline: torch.Tensor, degree: int) -> tuple:
    """
    Vectorized forward spline basis computation on CPU.
    
    Args:
        pseudo: Tensor of shape (E, D) - pseudo coordinates
        kernel_size: Tensor of shape (D,) - kernel size per dimension
        is_open_spline: Tensor of shape (D,) - whether spline is open per dimension
        degree: int - degree of the spline basis
    
    Returns:
        tuple: (basis, weight_index)
            - basis: Tensor of shape (E, S) - basis values
            - weight_index: Tensor of shape (E, S) - weight indices
    """
    # Input validation (same as before)
    assert pseudo.device.type == 'cpu', "pseudo must be on CPU"
    assert kernel_size.dim() == 1, "kernel_size must be 1D"
    assert pseudo.size(1) == kernel_size.numel(), "pseudo.size(1) must equal kernel_size.numel()"
    
    E = pseudo.size(0)
    D = pseudo.size(1)
    S = int(math.pow(degree + 1, D) + 0.5)
    
    basis = torch.ones((E, S), dtype=pseudo.dtype, device=pseudo.device)
    weight_index = torch.zeros((E, S), dtype=torch.int64, device=pseudo.device)
    
    # Process each dimension
    s_indices = torch.arange(S, device=pseudo.device)
    wi_offset = 1
    
    for d in range(D):
        # Compute k_mod for all s
        k_mod = s_indices % (degree + 1)
        s_indices = s_indices // (degree + 1)
        
        # Scale pseudo values
        v = pseudo[:, d:d+1] * (kernel_size[d] - degree * is_open_spline[d])  # (E, 1)
        
        # Compute weight index contribution
        wi_contrib = ((v + k_mod).long() % kernel_size[d]) * wi_offset
        weight_index += wi_contrib
        wi_offset *= kernel_size[d]
        
        # Compute basis values
        v = v - torch.floor(v)
        b_vals = torch.zeros_like(v).expand(E, S)
        
        for k_m in range(degree + 1):
            mask = (k_mod == k_m)
            b_vals[:, mask] = basis_function_forward_torch(
                v[mask], k_m, degree
            )
        
        basis *= b_vals
    
    return basis, weight_index


def basis_function_forward_torch(v: torch.Tensor, k_mod: int, degree: int) -> torch.Tensor:
    """
    Vectorized B-spline basis function using PyTorch operations.
    
    Args:
        v: Tensor - parameter values in [0, 1)
        k_mod: int - degree modulo
        degree: int - degree of the spline
    
    Returns:
        Tensor - basis function values
    """
    if degree == 2:
        if k_mod == 0:
            return 0.5 * v * v - v + 0.5
        elif k_mod == 1:
            return -v * v + v + 0.5
        else:  # k_mod == 2
            return 0.5 * v * v
    
    elif degree == 3:
        if k_mod == 0:
            return torch.pow(1.0 - v, 3) / 6.0
        elif k_mod == 1:
            return (3.0 * v ** 3 - 6.0 * v ** 2 + 4.0) / 6.0
        elif k_mod == 2:
            return (-3.0 * v ** 3 + 3.0 * v ** 2 + 3.0 * v + 1.0) / 6.0
        else:  # k_mod == 3
            return v ** 3 / 6.0
    
    else:
        raise NotImplementedError(f"Degree {degree} not implemented")


# Example usage:
if __name__ == "__main__":
    # Create sample data
    E, D = 10, 2  # 10 samples, 2 dimensions
    degree = 3
    
    pseudo = torch.randn(E, D, dtype=torch.float32)
    kernel_size = torch.tensor([5, 5], dtype=torch.int64)
    is_open_spline = torch.tensor([0, 0], dtype=torch.uint8)
    
    # Compute basis
    basis, weight_index = spline_basis_fw_cpu(pseudo, kernel_size, is_open_spline, degree)
    
    print(f"Basis shape: {basis.shape}")
    print(f"Weight index shape: {weight_index.shape}")
    print(f"Basis values:\n{basis}")
    print(f"Weight indices:\n{weight_index}")

k initial =  0
k_mod =  0
k =  0
k_mod =  0
k =  0
k initial =  1
k_mod =  1
k =  0
k_mod =  0
k =  0
k initial =  2
k_mod =  2
k =  0
k_mod =  0
k =  0
k initial =  3
k_mod =  3
k =  0
k_mod =  0
k =  0
k initial =  4
k_mod =  0
k =  1
k_mod =  1
k =  0
k initial =  5
k_mod =  1
k =  1
k_mod =  1
k =  0
k initial =  6
k_mod =  2
k =  1
k_mod =  1
k =  0
k initial =  7
k_mod =  3
k =  1
k_mod =  1
k =  0
k initial =  8
k_mod =  0
k =  2
k_mod =  2
k =  0
k initial =  9
k_mod =  1
k =  2
k_mod =  2
k =  0
k initial =  10
k_mod =  2
k =  2
k_mod =  2
k =  0
k initial =  11
k_mod =  3
k =  2
k_mod =  2
k =  0
k initial =  12
k_mod =  0
k =  3
k_mod =  3
k =  0
k initial =  13
k_mod =  1
k =  3
k_mod =  3
k =  0
k initial =  14
k_mod =  2
k =  3
k_mod =  3
k =  0
k initial =  15
k_mod =  3
k =  3
k_mod =  3
k =  0
k initial =  0
k_mod =  0
k =  0
k_mod =  0
k =  0
k initial =  1
k_mod =  1
k =  0
k_mod =  0
k =  0
k initial =  2
k_mod =  2
k =  0
k_mod =  0
k =  0
k initial =  3
k_mod =  3